# Quant Research Workflow

This notebook demonstrates the public, original version of the project workflow:

1. generate/load OHLCV data
2. build lagged features
3. create direction labels
4. run walk-forward predictions
5. backtest after transaction costs
6. evaluate risk metrics


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt

from quant_lab.data import MarketConfig, generate_synthetic_ohlcv
from quant_lab.features import make_model_frame
from quant_lab.validation import WalkForwardSplit
from quant_lab.models import make_random_forest_model, walk_forward_predict
from quant_lab.backtest import run_backtest
from quant_lab.metrics import performance_summary


In [ ]:
df = generate_synthetic_ohlcv(MarketConfig(n_days=2500, seed=42))
df.head()


In [ ]:
X, y = make_model_frame(df, horizon=5, threshold=0.002)
X.shape, y.value_counts(normalize=True).sort_index()


In [ ]:
splitter = WalkForwardSplit(train_size=750, test_size=125)
model = make_random_forest_model(random_state=42)
preds = walk_forward_predict(model, X, y, splitter)
preds.value_counts().sort_index()


In [ ]:
bt = run_backtest(df["close"], preds, transaction_cost_bps=5.0)
summary = performance_summary(bt["strategy_return"], bt["equity"])
pd.Series(summary).round(4)


In [ ]:
bt["equity"].plot(figsize=(10, 4), title="Strategy Equity Curve")
plt.ylabel("Equity")
plt.show()


## Interpretation

A good quant GitHub project should discuss limitations honestly.

Things to check before believing any result:

- Are features shifted properly?
- Is the validation time-aware?
- Are transaction costs included?
- Does performance survive different regimes?
- Is the strategy still attractive after turnover and drawdown?
